# HiveMind Agentic Benchmark: 4-Harness Arena (Core Suite & Polyglot Python)

Hardware-Optionen: **Tesla T4 (16 GB)**, **L4 (24 GB)** oder **A100 (40/80 GB)**, High-RAM, >=100 GB Disk.

---

## 1. Die drei Benchmark-Ziele

| Ziel | Hardware | Modell / Kontext | Suites & Tasks | Getestete Harnesses | Laufzeit-Erwartung |
|---|---|---|---|---|---|
| **Goal 1: Small-GPU-Baseline** | Tesla T4 (16 GB) / L4 (24 GB) | `qwen3.5:4b-mtp`<br>Context: **100k** (102.400 Tokens) | **23 Core Tasks** + **34 Polyglot Tasks**<br>3 Runs für statistische Varianz | `hivemind_duo`, `hivemind_duo_lean`, `aider`, `hermes`, `pi` | 1 Run (2 Harnesses): ~2h<br>1 Run (4 Harnesses): ~5-7h<br>3 Runs (4-5 Harnesses): **8–25h** |
| **Goal 2: Local Hardware Ceiling** | L4 (24 GB) / A100 (40–80 GB) | `qwen3.6:35b-a3b-ud`<br>Context: **100k** (102.400 Tokens) | **23 Core Tasks** + **34 Polyglot Tasks** | `hivemind_duo`, `hivemind_duo_lean`, `aider`, `hermes`, `pi` | 1 Run (2 Harnesses): ~2.5h<br>1 Run (4 Harnesses): ~6-9h |
| **Goal 3: DeepSWE Real-World** | A100 (40–80 GB) | `qwen3.6:35b-a3b-ud`<br>Context: **128k** (131.072 Tokens) | **3 echte Repo-Tasks** (`tomlkit`, `httpx`, `cattrs`), 1 Run | `hivemind_duo`, `hivemind_duo_lean`, `aider`, `hermes`, `pi` | ~1.5–2h (1800s Timeout pro Task) |

> [!NOTE]
> **4B & 35B mit 100k auf Colab:** Qwen 3.5 4B MTP nutzt Hybrid-Attention (nur 8 Attention-Layers) und läuft mit `q4_0` KV-Cache bei 100k (`102400` Tokens) sauber im T4-Budget. Bei knapperem VRAM oder Multi-Slot-Tests lässt sich `HM_CTX` anpassen (Hinweis: `hermes` braucht `>=65536`); `131072` (128k) bleibt für DeepSWE (35B) auf A100.
>
> **DeepSWE:** Es handelt sich um 3 reale Python-Repository-Bugfixes (nicht die vollen 113 Tasks von DeepSWE), damit der Lauf in realistischer Zeit und ohne Docker abgeschlossen werden kann.

---

## 2. Getestete HiveMind Toolsets & Competitor Harnesses

Wir vergleichen nicht Modelle, sondern **Agenten-Architekturen & Harnesses** auf exakt demselben Modell und `llama-server`:

### HiveMind Toolsets:
1. **`hivemind_duo` (Standard: `duo_core`, 19 Tools):**
   - **File & AST:** `read_file`, `get_signatures`, `find_references`, `list_dir`, `find_files`, `search_code`, `lsp_diagnostics`
   - **Edits:** `edit_file`, `apply_patch`, `write_file`, `write_file_append`, `undo_last`
   - **Execution & Verification:** `run_bash`, `run_python`, `run_tests`
   - **Interaction & Completion:** `task_complete`, `ask_user`, `get_datetime`, `send_file`
2. **`hivemind_duo_full` (Full Toolset, 41 Tools inkl. Optionals):**
   - Alle 19 Core-Tools PLUS vollständige LSP-Suite (`lsp_status`, `lsp_definition`, `lsp_references`, `lsp_hover`, `lsp_symbols`, `lsp_workspace_symbols`, `lsp_code_actions`, `lsp_format`, `lsp_rename`), Background-Prozesse (`start_background`, `get_background_output`, `stop_background`), `browser`, `subagent_research`, `install_package`, `remember`, `replace_lines`, `edit_ast` sowie (wenn aktiviert) `git_status`/`git_commit` und `web_search`/`web_fetch`.

### Competitor Harnesses:
3. **`aider`:** Offizielle Aider CLI (Diff Engine, Repo-Map via tree-sitter).
4. **`hermes`:** Nous Research Hermes Agent CLI (`NousResearch/hermes-agent`).
5. **`pi`:** Mario Zechner Pi Coding Agent CLI (`earendil-works/pi`).

> [!IMPORTANT]
> **Token-Metriken:** HiveMind (`usage_meta`), Aider (`stdout`), Hermes (`state.db`) und Pi (`--mode json`) erfassen alle ihre Token-Zahlen. Bei Aider wird intern vom CLI kein `cached_tokens`-Wert ausgegeben (steht dort auf `0%`), daher ist für alle Harnesses neben der **Pass-Rate (Resolved %)** und **Wall-Clock Time (Sekunden)** auch **Net-Tokens** vergleichbar (und bei `tok/s` zu beachten, dass KV-Cache-Hits den Brutto-Durchsatz erhöhen).

---

## 3. Empfohlene Ausführungs-Reihenfolge (Best Practice)

1. **Schritt 1 (Sanity Check):** Zelle 5 (Smoke Test: 3 Tasks, HiveMind vs. Aider) ~5-10 min.
2. **Schritt 2 (Phase 1 Sanity):** Zelle 6a mit `HM_CONDITIONS='hivemind_duo,aider'`, `HM_RUNS=1` (~30-45 min).
3. **Schritt 3 (Phase 1 Arena):** Zelle 6a mit `HM_CONDITIONS='hivemind_duo,aider,hermes,pi'`, `HM_RUNS=1` (~1.5h).
4. **Schritt 4 (Phase 2 Polyglot):** Zelle 7b mit 1 Run (~2-4h).
5. **Schritt 5 (Phase 3 DeepSWE):** Zelle 8b & 8c nur auf A100 mit 35B / 128k (~1.5h).
6. **Schritt 6 (Varianz-Lauf):** Wenn alle Smoke- und Einzelläufe grün sind, `HM_RUNS=3` für 3 Durchläufe starten.

In [ ]:
# 1) GPU & Disk prüfen
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1

In [ ]:
# 2) Repo aktualisieren, Google Drive verbinden & Checkpoints vorab syncen
import shutil, subprocess
from pathlib import Path

subprocess.run([
    "bash", "-lc",
    "set -e; [ -d /root/HiveMind_llama/.git ] || git clone --branch bench-stable https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama; "
    "cd /root/HiveMind_llama && git fetch origin bench-stable && git checkout bench-stable && git reset --hard origin/bench-stable && "
    "ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh && chmod +x /root/rent_bootstrap.sh && "
    "git log -1 --format='[repo] %h %s'"
], check=True)

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

DRIVE_RES = Path("/content/drive/MyDrive/hivemind_bench/results")
if Path("/content/drive/MyDrive").exists():
    DRIVE_RES.mkdir(parents=True, exist_ok=True)
    local_res = Path("/root/HiveMind_llama/bench/results")
    local_res.mkdir(parents=True, exist_ok=True)
    synced = 0
    for src in sorted(DRIVE_RES.glob("*.json")):
        dst = local_res / src.name
        if (not dst.exists()) or src.stat().st_size >= dst.stat().st_size:
            shutil.copy2(src, dst)
            synced += 1
            print(f"[pre-sync] {src.name} -> {dst}")
    print(f"[drive] Bereit ({DRIVE_RES}) | pre-synced: {synced} Checkpoints")
else:
    print("[drive] Nicht in Colab / kein Drive gemountet — Ergebnisse bleiben lokal unter /root/HiveMind_llama/bench/results")

In [ ]:
# 3) Basis-Setup (~2-3 min: Python 3.14, llama.cpp, Aider, Hermes & Pi installieren)
import subprocess
subprocess.run([
    "bash", "-lc",
    "set -e; ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh && chmod +x /root/rent_bootstrap.sh && HM_MODEL=skip HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup"
], check=True)

In [ ]:
# 3a) [DOWNLOAD OPTION A] 35B Modell herunterladen (~22 GB, ~10-15 min)
# Qwen 3.6 35B A3B UD-Q4_K_XL (MoE High-Performance Coder)
!bash /root/rent_bootstrap.sh download qwen3.6:35b-a3b-ud

In [ ]:
# 3b) [DOWNLOAD OPTION B] 4B Modell herunterladen (~3 GB, nur ~1-2 min)
# Qwen 3.5 4B MTP (Kompaktes 64k Modell - spart VRAM & Kosten, läuft auf jeder GPU wie T4)
!bash /root/rent_bootstrap.sh download qwen3.5:4b-mtp

In [ ]:
# 4a) [SERVER OPTION 1] 35B Modell mit 100k Kontext in den VRAM laden & starten
# Qwen 3.6 35B MoE | Auto-MoE-Budget (A100: 0 CPU, L4: ~4 CPU, T4: ~21 CPU, llama_mlock=False)
import os
os.environ["HM_MODEL"] = "qwen3.6:35b-a3b-ud"
os.environ["HM_CTX"] = "102400"

!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.6:35b-a3b-ud" HM_CTX=102400 bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

In [ ]:
# 4b) [SERVER OPTION 2] 4B Modell mit 100k Kontext in den VRAM laden & starten
# Qwen 3.5 4B MTP | Auto-Download falls fehlt (~2 Min) | T4/L4/A100 Standard: 100k (HM_CTX=102400)
import os
os.environ["HM_MODEL"] = "qwen3.5:4b-mtp"
os.environ["HM_CTX"] = "102400"

!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.5:4b-mtp" HM_CTX=102400 bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

In [ ]:
# 4c) VRAM- & Modell-Status prüfen
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
!echo "Aktives Modell:  $(cat /root/HiveMind_llama/.active_model 2>/dev/null || echo 'nicht gesetzt')"
!echo "Aktiver Kontext: $(cat /root/HiveMind_llama/.active_ctx 2>/dev/null || echo 'nicht gesetzt') Tokens"

In [ ]:
# 5) Smoke-Test (3 Tasks: HiveMind vs. Aider, ~5-10 min)
import subprocess
subprocess.run(["bash", "-lc", "set -e; ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh && chmod +x /root/rent_bootstrap.sh && bash /root/rent_bootstrap.sh smoke"], check=True)

In [ ]:
# 5b) OPTIONAL: Bestehende Benchmarks löschen (für sauberen Neustart ohne Resume/Merge)
DELETE_EXISTING_BENCHMARKS = False  # Zum Löschen auf True setzen!
import pathlib as _reset_pl
_reset_dirs = [
    _reset_pl.Path('/root/HiveMind_llama/bench/results'),
    _reset_pl.Path('/root/HiveMind_llama/bench/results/external'),
    _reset_pl.Path('/content/drive/MyDrive/hivemind_bench/results'),
    _reset_pl.Path('/content/drive/MyDrive/hivemind_bench/results/external'),
    _reset_pl.Path('/content/drive/MyDrive/hivemind_bench/logs'),
]
_reset_files = [f for d in _reset_dirs if d.exists() for f in (*d.glob('*.json'), *d.glob('*.log'))]
for _lf in (_reset_pl.Path('/root/HiveMind_llama/bench_full.log'), _reset_pl.Path('/root/HiveMind_llama/bench_engine.log')):
    if _lf.exists(): _reset_files.append(_lf)
if not DELETE_EXISTING_BENCHMARKS:
    print(f'[reset] ÜBERSPRUNGEN ({len(_reset_files)} Dateien behalten). Zum Löschen DELETE_EXISTING_BENCHMARKS = True setzen.')
else:
    for _f in _reset_files: _f.unlink()
    print(f'[reset] {len(_reset_files)} Benchmark-Dateien & Logs gelöscht — frischer Start bereit.')

---
## PHASE 1: HiveMind Core Suite (23 Tasks)
Testet Multi-File-Navigation, Refactoring, AST-Symbolerhalt und Selbstkorrektur.
Head-to-Head: HiveMind (`duo_core` oder `duo_full`) vs. `aider`, `hermes`, `pi` auf identischem Modell.

In [ ]:
# 6a) Phase 1 starten: Core Suite (23 Tasks)
# Nutzt automatisch das aktive Modell aus Zelle 4a (35B) oder 4b (4B)
# Wähle Bedingungen:
#   - Schnell / Sanity: 'hivemind_duo,aider' (~30-45 min)
#   - Arena Core + Lean: 'hivemind_duo,hivemind_duo_lean,aider,hermes,pi'
#   - Inkl. Full Toolset: 'hivemind_duo,hivemind_duo_lean,hivemind_duo_full,aider,hermes,pi'
import os, subprocess
os.environ['HM_RUNS'] = os.environ.get('HM_RUNS', '1')  # Standard '1' für Sanity Check; setze auf '3' für finale statistische Varianz
os.environ['HM_RUN_OFFSET'] = os.environ.get('HM_RUN_OFFSET', '0')  # Bei separaten Einzelläufen: Run 1='0', Run 2='1', Run 3='2'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,hivemind_duo_lean,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks'
os.environ.setdefault('HM_RESUME', '1')
subprocess.run(['bash', '/root/rent_bootstrap.sh', 'fg'], check=True)

In [ ]:
# 6b) Optional: Claude Code auf derselben Core Suite ausführen (braucht ANTHROPIC_API_KEY)
import os, shutil, subprocess
from pathlib import Path
if os.environ.get('ANTHROPIC_API_KEY'):
    subprocess.run(['/root/HiveMind_llama/.venv/bin/python', 'bench/external/run_external.py', '--harness', 'claude', '--tasks', 'bench/tasks'], cwd='/root/HiveMind_llama', check=True)
    if Path('/content/drive/MyDrive').exists():
        ext_dir = Path('/content/drive/MyDrive/hivemind_bench/results/external'); ext_dir.mkdir(parents=True, exist_ok=True)
        for f in Path('/root/HiveMind_llama/bench/results/external').glob('*.json'):
            shutil.copy2(f, ext_dir / f.name)
            print(f'[drive] Saved external -> {ext_dir / f.name}')
else:
    print('Kein ANTHROPIC_API_KEY gesetzt — Claude Code wird übersprungen.')

In [ ]:
# 6c) Live-Status der Phase 1 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard (aktualisiert alle 10s; Interrupt stoppt nur die Anzeige):
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py

---
## PHASE 2: Polyglot Python Practice Benchmark (34 Tasks)
Das offizielle Benchmark-Set von Aider AI (`Aider-AI/polyglot-benchmark/python/exercises/practice`).
Testet den kompletten 34-Aufgaben-Corpus im direkten Head-to-Head auf demselben llama-server.
Laufzeit: 1 Run mit 2 Harnesses ca. 1.5–2.5h; 1 Run mit 4 Harnesses ca. 4–6h.

In [ ]:
# 7a) Alle 34 Polyglot-Python-Practice-Aufgaben von Aider AI herunterladen und importieren
!HM_POLYGLOT_NAMES=all bash /root/rent_bootstrap.sh polyglot

In [ ]:
# 7b) Phase 2 starten: Polyglot Python Practice (34 Tasks)
# Nutzt automatisch das Modell aus Zelle 4a (35B) oder 4b (4B)
# Wähle Bedingungen:
#   - Schnell / Sanity: 'hivemind_duo,aider' (~1.5-2.5h)
#   - Arena Core + Lean: 'hivemind_duo,hivemind_duo_lean,aider,hermes,pi'
#   - Inkl. Full Toolset: 'hivemind_duo,hivemind_duo_lean,hivemind_duo_full,aider,hermes,pi'
import os, subprocess
os.environ['HM_RUNS'] = os.environ.get('HM_RUNS', '1')  # Standard '1'; setze auf '3' für 3 Durchläufe (Achtung: 3 Runs x 4 Harnesses = 408 Durchläufe, ~12-18h!)
os.environ['HM_RUN_OFFSET'] = os.environ.get('HM_RUN_OFFSET', '0')  # Bei separaten Einzelläufen: Run 1='0', Run 2='1', Run 3='2'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,hivemind_duo_lean,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks_polyglot'
os.environ.setdefault('HM_RESUME', '1')
subprocess.run(['bash', '/root/rent_bootstrap.sh', 'fg'], check=True)

In [ ]:
# 7c) Optional: Claude Code auf dem vollen Aider-Benchmark ausführen
import os, shutil, subprocess
from pathlib import Path
if os.environ.get('ANTHROPIC_API_KEY'):
    subprocess.run(['/root/HiveMind_llama/.venv/bin/python', 'bench/external/run_external.py', '--harness', 'claude', '--tasks', 'bench/tasks_polyglot'], cwd='/root/HiveMind_llama', check=True)
    if Path('/content/drive/MyDrive').exists():
        ext_dir = Path('/content/drive/MyDrive/hivemind_bench/results/external'); ext_dir.mkdir(parents=True, exist_ok=True)
        for f in Path('/root/HiveMind_llama/bench/results/external').glob('*.json'):
            shutil.copy2(f, ext_dir / f.name)
            print(f'[drive] Saved external -> {ext_dir / f.name}')
else:
    print('Kein ANTHROPIC_API_KEY gesetzt — Claude Code wird übersprungen.')

In [ ]:
# 7d) Live-Status der Phase 2 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard:
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py

---
## PHASE 3: DeepSWE Real-World Benchmark (Echte Repositories, 128k Kontext)
Schwere, reale Software-Engineering-Aufgaben aus `datacurve-ai/deep-swe` (`tomlkit`, `httpx`, `cattrs`).
Hier wird die volle Kontextstärke (128k / 131.072 Tokens) über alle Harnesses (`hivemind_duo`, `aider`, `hermes`, `pi`) getestet.
**Hardware-Voraussetzung:** Nur A100 GPU (40/80 GB) + 35B Modell; nicht auf L4/T4 ausführen.
VRAM-Bedarf bei 128k: ~33 GB VRAM (passt perfekt in die Colab A100 mit 40/80 GB).

In [ ]:
# 8a) DeepSWE Python-Tasks importieren & Test-Deps in .venv installieren
import subprocess
subprocess.run(['bash', '/root/rent_bootstrap.sh', 'deepswe'], check=True)

In [ ]:
# 8b) Engine mit 128k Kontext neu starten (~1 min)
# Nur auf A100 (40/80 GB) ausführen!
import os
os.environ["HM_MODEL"] = "qwen3.6:35b-a3b-ud"
os.environ["HM_CTX"] = "131072"

!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.6:35b-a3b-ud" HM_CTX=131072 bash /root/rent_bootstrap.sh serve
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

In [ ]:
# 8c) Phase 3 starten: DeepSWE Real-World Tasks (128k Kontext)
# 3 echte Repository-Tasks (tomlkit, httpx, cattrs)
import os, subprocess
os.environ['HM_RUNS'] = os.environ.get('HM_RUNS', '1')  # Bei DeepSWE (1800s Timeout pro Task) empfiehlt sich 1 Durchlauf
os.environ['HM_RUN_OFFSET'] = os.environ.get('HM_RUN_OFFSET', '0')  # Bei separaten Einzelläufen: Run 1='0', Run 2='1'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,hivemind_duo_lean,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks_deepswe'
os.environ['HM_CTX'] = '131072'
os.environ['HM_CELL_TIMEOUT'] = '1800'
os.environ.setdefault('HM_RESUME', '1')
subprocess.run(['bash', '/root/rent_bootstrap.sh', 'fg'], check=True)

In [ ]:
# 8d) Live-Status der Phase 3 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard:
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py

---
## AUSWERTUNG & REPORT (Das vereinte Leaderboard)
Analysiert die Ergebnisse wissenschaftlich (Wilson 95% CI, McNemar-Test, Net-Token-Ratio vs. Aider).

In [ ]:
# 9) Unified Leaderboard generieren
# 9a) Getrennt nach Modell & Harness (z. B. hivemind_duo@qwen3.6:35b-a3b-ud vs. hivemind_duo@qwen3.5:4b-mtp):
!cd /root/HiveMind_llama && .venv/bin/python bench/analyze.py --by-model

# 9b) Gepoolt über alle Runs hinweg (mit Wilson 95% CI und McNemar-Test):
# !cd /root/HiveMind_llama && .venv/bin/python bench/analyze.py

In [ ]:
# 10) Alle JSON-Ergebnisse & Logs vollständig in Google Drive sichern + verifizieren
import json, shutil
from pathlib import Path
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    d_root = Path('/content/drive/MyDrive/hivemind_bench')
    d_res, d_ext, d_logs = d_root / 'results', d_root / 'results' / 'external', d_root / 'logs'
    for d in (d_res, d_ext, d_logs): d.mkdir(parents=True, exist_ok=True)
    saved = 0
    for src_dir, dst_dir in [(Path('/root/HiveMind_llama/bench/results'), d_res), (Path('/root/HiveMind_llama/bench/results/external'), d_ext)]:
        for f in sorted(src_dir.glob('*.json')):
            shutil.copy2(f, dst_dir / f.name)
            rows = json.loads(f.read_text(encoding='utf-8')).get('results', [])
            ok = sum(1 for r in rows if r.get('ok'))
            print(f'[saved] {f.name:30s} | rows={len(rows):3d} ({ok} PASS) | {f.stat().st_size/1024:.1f} KB -> {dst_dir}')
            saved += 1
    for lf in ('bench_full.log', 'bench_engine.log'):
        p = Path('/root/HiveMind_llama') / lf
        if p.exists(): shutil.copy2(p, d_logs / lf)
    print(f'[drive] Sicherung abgeschlossen: {saved} JSON-Dateien unter {d_res}')
except ImportError:
    print('Nicht in Google Colab — Ergebnisse liegen unter /root/HiveMind_llama/bench/results/')

In [ ]:
# 11) Server stoppen
!bash /root/rent_bootstrap.sh stop